In [91]:
%matplotlib widget
import scipp as sc
from chemformula import ChemFormula

from redcamel import RemiCalculator, sample_photoionization

In [92]:
remi = RemiCalculator(
    length_acceleration_ion=sc.scalar(0.1, unit="m"),
    length_drift_ion=sc.scalar(0.0, unit="m"),
    voltage_ion=sc.scalar(10, unit="V"),
    length_acceleration_electron=sc.scalar(0.2, unit="m"),
    length_drift_electron=sc.scalar(0.0, unit="m"),
    voltage_electron=sc.scalar(20, unit="V"),
    magnetic_field=sc.scalar(3.5, unit="G"),
    v_jet=sc.scalar(1000, unit="m/s"),
    jet_direction="+x",
    field_direction="+z",
)

In [93]:
coincidence = sample_photoionization(
    atom_formula=ChemFormula("He"),
    binding_energy=sc.scalar(25, unit="eV"),
    photon_energy=sc.scalar(30, unit="eV"),
    energy_width=sc.scalar(1, unit="eV"),
    sizes={"pulses": 100_000, "p": 1},
    remi=remi,
)
coincidence.calculate_detector_hits()
helium_ion = coincidence.datagroup["He"]
electron = coincidence.datagroup["e"]

In [94]:
R_limit = sc.scalar(30, unit="mm")
pos_bins = 100
x_bins = sc.linspace("x", 0*R_limit, R_limit, pos_bins)
tof_bins = sc.linspace("tof", 0e4, 1.2e4, 1000, unit="ns")
x_tof_hist = helium_ion.hist(x=x_bins, tof=tof_bins, dim=("p", "pulses"))
x_tof_hist.plot(norm="log", cmap="PuBuGn")

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [98]:
R_limit = sc.scalar(60, unit="mm")
pos_bins = 100
R_bins = sc.linspace("R", 0*R_limit, R_limit, pos_bins)
tof_bins = sc.linspace("tof", 0, 8e2, 1000, unit="ns")
R_tof_hist = electron.hist(R=R_bins, tof=tof_bins, dim=("p", "pulses"))
R_tof_hist.plot(norm="log", cmap="PuBuGn")

InteractiveFigure(children=(HBar(), HBar(children=(VBar(children=(Toolbar(children=(ButtonTool(icon='home', la…

In [99]:
coincidence.electrons

{'e': <redcamel.remi_particles.Electron at 0x76c00ae081b0>}

In [115]:
electron.coords["x"] -= sc.scalar(30, unit="mm")

In [116]:
electron_converter_graph = remi.make_graph_for_momentum_calculation(
    coincidence.electrons["e"].mass, coincidence.electrons["e"].charge
)
electron = electron.transform_coords(["energy"], graph=electron_converter_graph)
electron

<scipp.DataArray>
Dimensions: Sizes[pulses:100000, p:1, ]
Coordinates:
* R                         float64             [mm]  (pulses, p)  [29.5515, 32.5786, ..., 18.8288, 8.13293]
* energy                    float64             [eV]  (pulses, p)  [5.16596, 5.4295, ..., 4.94332, 3.40288]
  p                         vector3    [au momentum]  (pulses, p)  [(-0.592733, 0.00254997, 0.168382), (0.12188, 0.450598, -0.425637), ..., (0.509952, -0.219851, -0.234397), (0.151438, -0.430696, -0.204144)]
  p_jet                     float64    [au momentum]  (pulses, p)  [-0.592733, 0.12188, ..., 0.509952, 0.151438]
  p_long                    float64    [au momentum]  (pulses, p)  [0.168382, -0.425637, ..., -0.234397, -0.204144]
  p_obs                     vector3    [au momentum]  (pulses, p)  [(-0.592733, 0.00254997, 0.168382), (0.12188, 0.450598, -0.425637), ..., (0.509952, -0.219851, -0.234397), (0.151438, -0.430696, -0.204144)]
  p_trans                   float64    [au momentum]  (pulses, p)  [0.00254997, 0.450598, ..., -0.219851, -0.430696]
  p_x                       float64    [au momentum]  (pulses, p)  [-0.592733, 0.12188, ..., 0.509952, 0.151438]
  p_y                       float64    [au momentum]  (pulses, p)  [0.00254997, 0.450598, ..., -0.219851, -0.430696]
  p_z                       float64    [au momentum]  (pulses, p)  [0.168382, -0.425637, ..., -0.234397, -0.204144]
  tof                       float64             [ns]  (pulses, p)  [331.487, 146.875, ..., 187.994, 195.909]
  tof_accel                 float64             [ns]  (pulses, p)  [331.487, 146.875, ..., 187.994, 195.909]
  tof_drift                 float64             [ns]  (pulses, p)  [0, 0, ..., 0, 0]
  x                         float64             [mm]  (pulses, p)  [-50.957, 2.49079, ..., -48.7523, -34.5404]
  y                         float64             [mm]  (pulses, p)  [20.8351, -2.38986, ..., -1.69509, 6.74755]
Data:
                            float64  [dimensionless]  (pulses, p)  [1, 1, ..., 1, 1]